# Causal Inference: The Effect of Discounts on Purchase Probability

## Business Problem

Discounts are a common strategy for encouraging customers to make a purchase. However, customers may respond differently to the same discount, making it important to understand both the overall effect and how that effect varies across customer characteristics.

This project investigates how offering a discount affects purchase probability and whether the estimated effect differs across customers with different levels of purchase intent.

## Objectives

- **Estimate the treatment effect:** Estimate how a 10% discount affects purchase probability compared with no discount.
- **Explore treatment heterogeneity:** Investigate whether the estimated effect varies across customers with different levels of purchase intent.
- **Translate findings into business insights:** Discuss how causal effect estimates could inform discount strategies and what additional evidence would be needed before implementation.

## Methodology

The analysis uses synthetic customer data and `CausalForestDML`, a causal machine learning method, to estimate treatment effects while accounting for observed customer characteristics that may influence both discount assignment and purchase behaviour.

The `purchase_intent` feature represents a customer's underlying propensity to purchase, conceptually informed by historical platform behaviour, such as browsing activity, previous purchases, and engagement patterns. In this project, it is a synthetically generated feature rather than a score calculated from actual historical activity. It represents information that would ideally be available before a discount is assigned and the purchase outcome is observed.

## Important Notes

- **Synthetic data:** The dataset is artificially generated for educational and demonstration purposes. The findings do not represent actual customer behaviour.
- **Non-random discount assignment:** Customers are not randomly assigned discounts. Although the model accounts for observed customer characteristics, unobserved confounding may remain.
- **Business limitations:** An increase in purchase probability does not necessarily translate into increased profit. Product margins, discount costs, and potential changes in customer spending would need to be considered before implementing a discount strategy.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
results_df = pd.read_csv('ecommerce_discount_data.csv')
results_df


,customer_id,previous_orders,basket_value,days_since_last_order,website_visits,purchase_intent,discount_pct,purchased
0,1,4,74.86,171,4,0.727,28.54,0
1,2,4,46.62,31,8,0.218,0.00,0
2,3,5,79.78,3,3,0.494,13.62,0
3,4,1,61.17,21,5,0.414,0.00,0
4,5,7,36.37,125,3,0.652,42.74,1
...,...,...,...,...,...,...,...,...
9995,9996,1,73.83,66,2,0.829,7.73,0
9996,9997,3,43.44,158,4,0.675,0.00,0
9997,9998,1,94.06,66,4,0.565,40.54,0
9998,9999,5,53.17,67,6,0.478,0.00,0


### Data Overview and Quality Checks

Examine the dataset's dimensions, data types, missing values, duplicate records, and descriptive statistics.

In [3]:
display(results_df.shape)
results_df.info()
display(results_df.isnull().sum())
display(results_df.duplicated().sum())
display(results_df.describe().T)

(10000, 8)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            10000 non-null  int64  
 1   previous_orders        10000 non-null  int64  
 2   basket_value           10000 non-null  float64
 3   days_since_last_order  10000 non-null  int64  
 4   website_visits         10000 non-null  int64  
 5   purchase_intent        10000 non-null  float64
 6   discount_pct           10000 non-null  float64
 7   purchased              10000 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 625.1 KB


customer_id              0
previous_orders          0
basket_value             0
days_since_last_order    0
website_visits           0
purchase_intent          0
discount_pct             0
purchased                0
dtype: int64

np.int64(0)

,count,mean,std,min,25%,50%,75%,max
customer_id,10000.0,5000.500000,2886.895680,1.000,2500.7500,5000.500,7500.25000,10000.000
previous_orders,10000.0,3.010300,1.745308,0.000,2.0000,3.000,4.00000,11.000
basket_value,10000.0,64.551409,24.573634,5.000,47.7975,64.180,81.15250,190.180
days_since_last_order,10000.0,90.521600,51.951591,1.000,45.0000,91.000,135.00000,180.000
website_visits,10000.0,5.016200,2.251676,0.000,3.0000,5.000,6.00000,14.000
purchase_intent,10000.0,0.502075,0.225394,0.007,0.3270,0.501,0.67625,0.996
discount_pct,10000.0,13.787537,14.667582,0.000,0.0000,10.370,26.55000,49.820
purchased,10000.0,0.257800,0.437445,0.000,0.0000,0.000,1.00000,1.000


### Treatment and Outcome Checks
Examine the distribution of discount values, the number of customers receiving no discount versus a positive discount, and the purchase outcome.

In [4]:
print("Zero discount:", (results_df["discount_pct"] == 0).sum())
print("Positive discount:", (results_df["discount_pct"] > 0).sum())
print("Minimum discount:", results_df["discount_pct"].min())
print("Maximum discount:", results_df["discount_pct"].max())
print("Purchase rate:", results_df["purchased"].mean())
print("Outcome values:", sorted(results_df["purchased"].unique()))

Zero discount: 4501
Positive discount: 5499
Minimum discount: 0.0
Maximum discount: 49.82
Purchase rate: 0.2578
Outcome values: [np.int64(0), np.int64(1)]


### Observed Purchase Rates by Discount Band
Group customers by discount size to compare purchase rates across discount levels. These descriptive results show observed associations and do not establish causal effects.

In [3]:
results_df["discount_band"] = pd.cut(
    results_df["discount_pct"],
    bins=[-0.01, 0, 10, 20, 30, 40, 50],
    labels=["0%", "(0, 10%]", "(10, 20%]", "(20, 30%]",
            "(30, 40%]", "(40, 50%]"]
)

results_df.groupby("discount_band", observed=False)["purchased"].agg(
    customers="count",
    purchase_rate="mean"
).reset_index()

,discount_band,customers,purchase_rate
0,0%,4501,0.132859
1,"(0, 10%]",464,0.209052
2,"(10, 20%]",1353,0.288987
3,"(20, 30%]",1811,0.368305
4,"(30, 40%]",1430,0.430070
5,"(40, 50%]",441,0.476190


### Customer Characteristics by Discount Assignment
Compare average customer characteristics between those receiving no discount and those receiving a positive discount to identify potential differences in the groups.

In [4]:
results_df.groupby(
    results_df["discount_pct"] > 0
)[
    [
        "previous_orders",
        "basket_value",
        "days_since_last_order",
        "website_visits",
        "purchase_intent"
    ]
].mean().T.rename(
    columns={False: "No discount", True: "Positive discount"}
)

discount_pct,No discount,Positive discount
previous_orders,3.000444,3.018367
basket_value,62.234794,66.447587
days_since_last_order,90.055765,90.902891
website_visits,4.995112,5.033461
purchase_intent,0.473223,0.525690


### Define the Outcome, Treatment, and Features
Define purchase (purchased) as the outcome, discount size (discount_pct) as the treatment, and customer characteristics as the input features. Exclude identifiers and outcome/treatment-related columns from the feature set.

In [7]:
# Target
Y = results_df["purchased"]

# Treatment (discount size)
T = results_df["discount_pct"]

# Features
X = results_df.drop(
    columns=["customer_id", "purchased", "discount_pct", "discount_band"]
)

### Train-Test Split
Split the data into training (70%) and test (30%) sets. The training set is used to fit the causal model, while the test set allows us to examine estimated treatment effects on customers not used to train it.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X, T, Y, test_size=0.3, random_state=42
)

### Fit the Causal Forest Model
Fit a CausalForestDML model to estimate how discount size affects purchase probability while accounting for observed customer characteristics. The model also allows us to explore how the estimated treatment effect varies across customers.

In [9]:
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor

est = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, min_samples_leaf=10),
    model_t=RandomForestRegressor(n_estimators=200, min_samples_leaf=10),
    n_estimators=500,
    min_samples_leaf=5,
    random_state=42
)

est.fit(
    Y_train,
    T_train,
    X=X_train
)

### Estimated Effect of a 10% Discount
Estimate the effect of a 10% discount compared with no discount for each customer in the test set, then calculate the average estimated effect.

In [11]:
# Estimated effect of a 10% discount vs no discount
cate_10 = est.effect(X_test, T0=0, T1=10)

# Average treatment effect
ate_10 = cate_10.mean()

# Confidence interval for the average effect
lb_10, ub_10 = est.effect_interval(X_test, T0=0, T1=10)

print(f"Average effect: {ate_10:.4f}")
print(f"95% CI: [{lb_10.mean():.4f}, {ub_10.mean():.4f}]")

Average effect: 0.0716
95% CI: [0.0299, 0.1132]


The model estimates that a 10% discount increases purchase probability by an average of **7.18 percentage points** compared with no discount among customers in the test set.

This is a model-based estimate rather than a confirmed causal effect. Since discount assignment is not random, unobserved factors may still influence the results.

### Treatment Effect Heterogeneity by Purchase Intent
Group test-set customers into three purchase-intent categories and compare the average estimated effect of a 10% discount versus no discount across these groups.

In [12]:
# Explore how the estimated discount effect varies by purchase intent

heterogeneity_df = X_test[["purchase_intent"]].copy()
heterogeneity_df["estimated_effect"] = cate_10

heterogeneity_df["intent_group"] = pd.qcut(
    heterogeneity_df["purchase_intent"],
    q=3,
    labels=["Low", "Medium", "High"]
)

print(
    heterogeneity_df.groupby("intent_group", observed=True)
    ["estimated_effect"]
    .mean()
    .mul(100)
    .round(2)
    .rename("Estimated effect (percentage points)")
)


intent_group
Low       5.34
Medium    7.34
High      8.80
Name: Estimated effect (percentage points), dtype: float64


The model estimates a larger effect of a 10% discount among customers with higher purchase intent. The estimated effect increases from **5.34 percentage points** for the low-intent group to **7.34 percentage points** for the medium-intent group and **8.80 percentage points** for the high-intent group.

These exploratory results suggest that estimated discount responsiveness may vary across customers. However, these are model-based estimates, and the differences should be validated before informing real-world discount strategies.

### Define Comparison Groups for Observed Purchase Rates
Create two groups for a descriptive comparison: customers receiving no discount and customers receiving a discount of at least 10%. Customers receiving a discount below 10% are excluded from this comparison.

In [13]:
# Define treatment: received at least a 10% discount
analysis_df = results_df.copy()
analysis_df["treated"] = (
    analysis_df["discount_pct"] >= 10
).astype(int)

# Exclude customers receiving a discount below 10%
analysis_df = analysis_df[
    (analysis_df["treated"] == 1) |
    (analysis_df["discount_pct"] == 0)
]

print(analysis_df["treated"].value_counts().rename({
    0: "No discount",
    1: "10%+ discount"
}))


treated
10%+ discount    5035
No discount      4501
Name: count, dtype: int64


### Observed Purchase Rate Comparison
Compare purchase rates between customers receiving no discount and those receiving a discount of at least 10%, including the number of customers and purchases in each group.

In [14]:
# Compare observed purchase rates
observed_rates = (
    analysis_df.groupby("treated")["purchased"]
    .agg(["mean", "count", "sum"])
)

observed_rates.index = ["No discount", "10%+ discount"]
observed_rates["purchase_rate_pct"] = observed_rates["mean"] * 100

print(observed_rates[["count", "sum", "purchase_rate_pct"]].round(2))


               count   sum  purchase_rate_pct
No discount     4501   598              13.29
10%+ discount   5035  1883              37.40


The observed purchase rate was **37.40%** among customers receiving a discount of at least 10%, compared with **13.29%** among customers receiving no discount — an observed difference of **24.11 percentage points**.

However, this is an unadjusted observational comparison, not a causal estimate. Customers were not randomly assigned discounts, and the discounted group may differ in other characteristics. The causal model provides a separate, adjusted estimate for a 10% discount versus no discount.

## Conclusion

The causal forest model estimated that a 10% discount was associated with an average increase of **7.18 percentage points** in purchase probability compared with no discount among test-set customers. Estimated effects also varied across purchase-intent groups, suggesting that discount responsiveness may differ between customers.

The observed purchase-rate difference between discounted and non-discounted customers was larger, highlighting why raw comparisons should not be interpreted as causal effects.

These findings are illustrative because the data is synthetic and discount assignment is not random. Further validation, controlled experiments, and profitability analysis would be needed before using these estimates to guide real-world discount strategies.